```markdown
# 基于CNN的多标签图像识别与前景信息建模

本Notebook保留18类、809张图像的原任务。七种评价条件为普通GAP、残差GAP、GMP、A共享融合、B逐类融合、G2固定权重推理加权和G3训练加权。C及G1是普通GAP的别名。

六种训练方案各进行五折训练和一次687图重训，共36条训练轨迹；G2不训练，绑定本次运行的GAP权重。历史参考表与新运行输出分开保存。

**入口**
- results：默认。无需数据或GPU，只读取已核验的参考结果，不构建模型、不训练、不创建outputs。
- check：读取原数据，检查身份、同步几何、手算和小样本梯度；没有optimizer.step，也不评价历史测试集。
- train：先检查，再从头运行或恢复完整实验。正式训练使用CUDA，耗时明显长于检查。

主改动是空间汇总；训练损失仍为正例加权BCE，选模为验证Macro-F1，分类阈值只用验证集确定，最终同时报告mAP及F1。前景组使用原PNG的alpha额外信息，不是学习分割。全部结果边界见README。

本公开整理版没有重新完成36条完整训练；附带成绩来自2026年10月27日冻结产物，不预先保证跨硬件逐位相同。
```


In [ ]:
# =========================
# 1. 导入配置与运行模式
# =========================
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import io, json, copy, time, random, hashlib, platform
from pathlib import Path
from datetime import datetime, timezone, timedelta
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms
from torchvision.transforms import functional as vision_F
from torch.utils.data import Dataset, DataLoader
import sklearn
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score
from PIL import Image
import PIL
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display

MODE = "results"  # 只允许 results / check / train；默认不训练、不读取原始数据。
ROOT = Path.cwd().resolve()  # 从仓库根目录启动Jupyter，再打开本Notebook。
assert (ROOT / "config/experiment.json").is_file(), "请从仓库根目录启动Jupyter。"
DATA_DIR = Path(os.environ.get("POKEMON_DATA_DIR", str(ROOT / "data"))).expanduser().resolve()
OUTPUT_DIR = Path(os.environ.get("POKEMON_OUTPUT_DIR", str(ROOT / "outputs"))).expanduser().resolve()
NOTEBOOK_PATH = ROOT / "Pokemon_CNN_Comparison.ipynb"
REFERENCE_DIR = ROOT / "results/reference_20261027"
CONFIG = json.loads((ROOT / "config/experiment.json").read_text(encoding="utf-8"))
DATA_MANIFEST = json.loads((ROOT / "config/data_manifest.json").read_text(encoding="utf-8"))
FIXED_SPLITS = json.loads((ROOT / "config/splits.json").read_text(encoding="utf-8"))
REFERENCE = json.loads((ROOT / "config/reference.json").read_text(encoding="utf-8"))
CLASS_NAMES = DATA_MANIFEST["class_names"]
TRAIN_MODEL_NAMES = ["Plain_GAP", "Residual_GAP", "Plain_GMP", "A_Shared", "B_Classwise", "G3_Foreground"]
ALL_MODEL_NAMES = ["Plain_GAP", "Residual_GAP", "Plain_GMP", "A_Shared", "B_Classwise", "G2_Foreground", "G3_Foreground"]
THRESHOLDS = np.asarray(CONFIG["thresholds"], dtype=np.float64)  # 保留原np.arange产生的实际浮点网格。
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ENVIRONMENT = {"python": platform.python_version(), "torch": torch.__version__,
               "torchvision": torchvision.__version__, "cuda": torch.version.cuda,
               "numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__,
               "pillow": PIL.__version__, "matplotlib": matplotlib.__version__,
               "device": str(DEVICE), "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None}
assert MODE in ("results", "check", "train")
assert len(CLASS_NAMES) == 18 and CONFIG["top_k"] == 16 and CONFIG["dropout"] == 0.5
assert CONFIG["folds"] == 5 and CONFIG["epochs"] == 100  # 此版本固定正式比较口径。
assert OUTPUT_DIR != ROOT and OUTPUT_DIR != DATA_DIR
assert OUTPUT_DIR != (ROOT / "results") and (ROOT / "results") not in OUTPUT_DIR.parents
plt.rcParams["axes.unicode_minus"] = False
print("模式：", MODE, "；设备：", DEVICE)


```markdown
## 读取数据与固定划分

数据目录应包含pokemon.csv和images/。只使用Name、Type1、Type2三列，允许CSV换行编码或Evolution等未使用列变化，但必须通过三列内容与行序的语义摘要、全部图像逐文件摘要核验。下载页更新后若不匹配，先核查版本，不重新随机划分或跳过校验。

687张开发图与122张历史留出图固定；普通KFold划分开发集，前两折549/138，后三折550/137。数据校验会读图像身份及标签；check模式的模型小样本只来自开发集。

results模式不执行本节读取函数。
```


In [ ]:
# =========================
# 2. 数据身份与划分函数
# =========================
def hash_file(path):
    """计算文件摘要，用于来源校验和断点身份确认。"""
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def semantic_digest(table):
    """只绑定模型使用的名称、两列标签及行序，不把无用列算作输入。"""
    records = [[str(row.Name), str(row.Type1), None if pd.isna(row.Type2) else str(row.Type2)]
               for row in table.itertuples(index=False)]
    encoded = json.dumps(records, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()

def load_dataset():
    """只读原始数据，逐图检查身份；不访问历史实验目录。"""
    csv_path = DATA_DIR / "pokemon.csv"
    if not csv_path.is_file():
        raise FileNotFoundError("请按README准备pokemon.csv及images/，或设置POKEMON_DATA_DIR。")
    table = pd.read_csv(csv_path).reset_index(drop=True)
    assert {"Name", "Type1", "Type2"}.issubset(table.columns), "标签CSV缺少必要列"
    assert len(table) == DATA_MANIFEST["sample_count"] == 809
    assert semantic_digest(table) == DATA_MANIFEST["semantic_sha256"], "数据版本或标签行序不匹配，请核查下载文件。"
    table["row_id"] = np.arange(len(table), dtype=np.int64)
    table["image_id"] = table["Name"].str.lower() + ".png"
    table["image_path"] = table["image_id"].map(lambda name: str(DATA_DIR / "images" / name))
    assert table["image_id"].is_unique
    class_to_index = {name: index for index, name in enumerate(CLASS_NAMES)}
    label_matrix = np.zeros((len(table), len(CLASS_NAMES)), dtype=np.float32)
    records = []
    for row in table.itertuples(index=False):
        for type_name in (row.Type1, row.Type2):
            if pd.isna(type_name):
                continue  # 缺失的第二标签不形成新类别。
            label_matrix[row.row_id, class_to_index[type_name]] = 1.0
        image_bytes = Path(row.image_path).read_bytes()
        with Image.open(io.BytesIO(image_bytes)) as image:
            image.load()
            assert image.mode == "RGBA" and image.size == (120, 120), row.image_id
            assert image.getchannel("A").getbbox() is not None, row.image_id
        records.append({"row_id": row.row_id, "image_id": row.image_id,
                        "sha256": hashlib.sha256(image_bytes).hexdigest()})
    assert records == DATA_MANIFEST["image_records"], "图片内容或图片顺序与实验数据不一致。"
    assert len({item["sha256"] for item in records}) == 809
    assert int((label_matrix.sum(1) == 1).sum()) == 404
    assert int((label_matrix.sum(1) == 2).sum()) == 405
    return table, label_matrix, pd.DataFrame(records), hash_file(csv_path)

def load_fixed_splits(label_matrix):
    """使用随仓库保存的原始成员，同时复算划分以避免静默错位。"""
    dev_ids = np.asarray(FIXED_SPLITS["dev_ids"], dtype=np.int64)
    test_ids = np.asarray(FIXED_SPLITS["test_ids"], dtype=np.int64)
    expected_dev, expected_test = train_test_split(np.arange(809), test_size=CONFIG["test_ratio"],
                                                   random_state=CONFIG["seed"], shuffle=True)
    assert np.array_equal(dev_ids, expected_dev) and np.array_equal(test_ids, expected_test)
    folds = []
    for index, (train_rows, val_rows) in enumerate(KFold(5, shuffle=True, random_state=CONFIG["seed"]).split(dev_ids)):
        item = {"fold": index + 1, "seed": CONFIG["fold_seeds"][index],
                "train": dev_ids[train_rows], "val": dev_ids[val_rows]}
        expected = FIXED_SPLITS["folds"][index]
        assert item["train"].tolist() == expected["train"] and item["val"].tolist() == expected["val"]
        assert item["seed"] == expected["seed"]
        folds.append(item)
    assert len(dev_ids) == 687 and len(test_ids) == 122
    assert set(dev_ids).isdisjoint(test_ids)
    assert np.array_equal(np.sort(np.concatenate([fold["val"] for fold in folds])), np.sort(dev_ids))
    for ids in [dev_ids, test_ids] + [fold[key] for fold in folds for key in ("train", "val")]:
        assert np.all(label_matrix[ids].sum(0) > 0)
    return dev_ids, test_ids, folds


```markdown
## 辅助函数与同步图像处理

RGB保持原Resize128、随机水平翻转和±15度旋转，旋转NEAREST、不扩展画布、填0；评价关闭随机增强。alpha共用同一次几何抽样，不另抽随机数、不做RGB归一化。

随机流分离：训练全局流seed+4000，loader流+1000，增强流+3000，评价loader流+2000。断点同时恢复模型、BN、Adam及这些状态。保存的路径都相对本次输出目录，不读取本机旧项目。
```


In [ ]:
# =========================
# 3. 保存、随机状态与同步变换
# =========================
def now_local():
    """所有时间记录为北京时间。"""
    return datetime.now(timezone(timedelta(hours=8))).isoformat(timespec="seconds")

def seed_everything(seed):
    """仅新阶段重设全局随机流；恢复时使用保存状态。"""
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False
    torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False
    torch.use_deterministic_algorithms(True)

def cpu_state(model):
    """复制参数及BN缓冲区，最佳状态不随训练变化。"""
    return {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}

def state_digest(state):
    """逐项哈希完整模型参数和BN状态。"""
    h=hashlib.sha256()
    for k in sorted(state):
        h.update(k.encode("utf-8"));h.update(state[k].contiguous().cpu().numpy().tobytes())
    return h.hexdigest()

def save_json(path, value):
    """同目录临时文件写完后替换，避免留下半份JSON。"""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    os.replace(temporary, path)

def save_checkpoint(path, value):
    """完整保存模型、优化器和随机状态；仅train模式运行阶段调用。"""
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    torch.save(value, temporary)
    os.replace(temporary, path)

def implementation_hash():
    """记录公开Notebook代码身份，忽略模式选择和执行输出。"""
    notebook = json.loads(NOTEBOOK_PATH.read_text(encoding="utf-8"))
    sources = []
    for cell in notebook["cells"]:
        if cell["cell_type"] == "code":
            lines = [line for line in "".join(cell["source"]).splitlines() if not line.startswith("MODE = ")]
            sources.append("\n".join(lines))
    return hashlib.sha256("\n\n".join(sources).encode("utf-8")).hexdigest()

def relative_output(path):
    """产物来源使用输出目录内的相对路径，便于搬迁整个运行目录。"""
    return Path(path).resolve().relative_to(OUTPUT_DIR).as_posix()

def build_transforms():
    """保留第18节原transform，用于逐像素与随机状态核验。"""
    train_transform = transforms.Compose([
        transforms.Resize((128, 128)), transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15), transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    eval_transform = transforms.Compose([
        transforms.Resize((128, 128)), transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    return train_transform, eval_transform

class OriginalPairedDataset(Dataset):
    """保留第18节三元样本接口，只用于新旧输入配对预检。"""
    def __init__(self, indices, training=False, seed=42):
        """创建独立增强流；不改变全局随机状态。"""
        self.indices = np.asarray(indices, dtype=np.int64)
        self.training = training
        self.augmentation_generator = torch.Generator().manual_seed(seed + CONFIG['augmentation_rng_offset'])
    def __len__(self):
        """返回固定成员数量。"""
        return len(self.indices)
    def __getitem__(self, index):
        """按原规则返回RGB、标签和原行号。"""
        row = int(self.indices[index])
        image = RGB_IMAGES[row].copy()
        if self.training:
            with torch.random.fork_rng(devices=[]):
                torch.set_rng_state(self.augmentation_generator.get_state())
                tensor = TRAIN_TRANSFORM(image)
                self.augmentation_generator.set_state(torch.get_rng_state())
        else:
            tensor = EVAL_TRANSFORM(image)
        return tensor, torch.from_numpy(labels[row].copy()), row

def paired_transform(image, alpha, training=False):
    """只抽一次原几何参数，同时变换RGB和原8bit alpha；不裁框或合成背景。"""
    image, alpha = PAIR_RESIZE(image), PAIR_RESIZE(alpha)
    if training:
        if torch.rand(1) < 0.5:  # 与原RandomHorizontalFlip的抽样次序和调用一致。
            image, alpha = vision_F.hflip(image), vision_F.hflip(alpha)
        angle = transforms.RandomRotation.get_params([-15.0, 15.0])
        image = vision_F.rotate(image, angle, interpolation=transforms.InterpolationMode.NEAREST,
                                expand=False, center=None, fill=0)
        alpha = vision_F.rotate(alpha, angle, interpolation=transforms.InterpolationMode.NEAREST,
                                expand=False, center=None, fill=0)
    image_tensor = vision_F.normalize(vision_F.to_tensor(image),
                                     [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    mask = vision_F.to_tensor(alpha)  # PIL L的uint8值自动转float32并除255，不做RGB归一化。
    assert mask.shape == (1, 128, 128) and mask.dtype == torch.float32
    assert bool(torch.isfinite(mask).all()) and float(mask.min()) >= 0.0 and float(mask.max()) <= 1.0
    assert float(mask.sum()) > 0.0, '无有效前景，不允许悄悄回退GAP'
    return image_tensor, mask

class PairedForegroundDataset(Dataset):
    """返回原RGB、同步alpha、标签、行号；增强流与第18节保持相同推进量。"""
    def __init__(self, indices, training=False, seed=42):
        """创建与原数据集同初态的持久增强流。"""
        self.indices = np.asarray(indices, dtype=np.int64)
        self.training = training
        self.augmentation_generator = torch.Generator().manual_seed(seed + CONFIG['augmentation_rng_offset'])
    def __len__(self):
        """返回固定成员数量。"""
        return len(self.indices)
    def __getitem__(self, index):
        """临时切入增强流，退出后全局CPU RNG原样恢复。"""
        row = int(self.indices[index])
        image, alpha = RGB_IMAGES[row].copy(), ALPHA_IMAGES[row].copy()
        if self.training:
            with torch.random.fork_rng(devices=[]):
                torch.set_rng_state(self.augmentation_generator.get_state())
                tensor, mask = paired_transform(image, alpha, True)
                self.augmentation_generator.set_state(torch.get_rng_state())
        else:
            tensor, mask = paired_transform(image, alpha, False)
        return tensor, mask, torch.from_numpy(labels[row].copy()), row

def make_loader(dataset,seed,training=False,generator=None):
    """训练和评价使用相互独立的generator，保持原batch和drop_last。"""
    if generator is None:
        offset=CONFIG["loader_rng_offset"] if training else CONFIG["eval_rng_offset"]
        generator=torch.Generator().manual_seed(seed+offset)
    return DataLoader(dataset,batch_size=CONFIG["batch_size"],shuffle=training,num_workers=CONFIG["num_workers"],
                      drop_last=False,pin_memory=(DEVICE.type == "cuda"),generator=generator)

def compute_pos_weight(indices):
    """只使用当前训练成员计算正类权重，最终687阶段重新计算。"""
    positive=labels[indices].sum(0)
    return torch.tensor((len(indices)-positive)/np.maximum(positive,1.0),dtype=torch.float32)

def capture_random(loader_generator,dataset):
    """保存模型训练、样本顺序和几何增强的全部可恢复状态。"""
    return {"python":random.getstate(),"numpy":np.random.get_state(),"cpu":torch.get_rng_state(),
            "cuda":torch.cuda.get_rng_state_all(),"loader":loader_generator.get_state(),
            "augmentation":dataset.augmentation_generator.get_state()}

def restore_random(saved,loader_generator,dataset):
    """构造模型并加载状态后恢复，不用初始seed覆盖已运行随机流。"""
    random.setstate(saved["python"]);np.random.set_state(saved["numpy"])
    torch.set_rng_state(saved["cpu"]);torch.cuda.set_rng_state_all(saved["cuda"])
    loader_generator.set_state(saved["loader"])
    dataset.augmentation_generator.set_state(saved["augmentation"])

TRAIN_TRANSFORM, EVAL_TRANSFORM = build_transforms()
PAIR_RESIZE = transforms.Resize((128, 128))
PairedDataset = PairedForegroundDataset


```markdown
## 七种方案及初始化

- 普通GAP：所有特征位置等权平均；残差GAP使用原自建残差块；GMP先逐特征通道取最大值再分类。
- A/B：先生成18类得分图，全图均值g与Top16均值l按z=(1−β)g+βl融合。β=sigmoid(θ)，A共用一个θ，B有18个；不保证Top16对应语义部件。
- G2/G3：alpha缩小为16×16权重m，按h=Σ(mF)/Σm汇总，再进原Dropout及Linear；mask不学习，参数量不增加。

普通主干及分类器从同种子未训练状态配对；A/B使用Linear到1×1 Conv映射，G3与GAP初态相同。每条训练正式开始时重新设置训练随机流，因此模型构造本身不改变训练抽样起点。G2只绑定本次运行的GAP最佳/最终权重，不把参考成绩中的87轮硬编码为规则。
```


In [ ]:
# =========================
# 4. 模型定义与配对初始化
# =========================
class DeterministicGlobalMax(nn.Module):
    """单个全局窗口取max，等价原AdaptiveMaxPool2d(1)且支持确定性CUDA反传。"""
    def forward(self,x,mask=None):
        """并列最大值取空间展平后的第一个位置，与原全局max梯度一致。"""
        return x.flatten(2).max(dim=-1).values[:, :, None, None]

class ResidualBlock(nn.Module):
    """保留原自定义残差块，卷积分支和捷径分支相加。"""
    def __init__(self, in_c, out_c):
        """构造模型或数据对象，保持原参数与操作顺序。"""
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
        )
        self.shortcut = nn.Sequential()
        if in_c != out_c:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_c, out_c, 1),
                nn.BatchNorm2d(out_c),
            )

    def forward(self, x, mask=None):
        """按当前模型定义提取并汇总特征，输出未经sigmoid的logit。"""
        return F.relu(self.conv(x) + self.shortcut(x))

class PokemonModel(nn.Module):
    """原三阶段CNN主干，支持GAP或GMP汇总。"""
    def __init__(self, use_res=False, pool_type="GAP", num_classes=18):
        """构造模型或数据对象，保持原参数与操作顺序。"""
        super().__init__()

        def make_block(in_c, out_c):
            """构造当前通道阶段的两次卷积或残差分支。"""
            if use_res:
                return ResidualBlock(in_c, out_c)
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(),
                nn.Conv2d(out_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(),
            )

        self.enc = nn.Sequential(
            make_block(3, 32), nn.MaxPool2d(2),
            make_block(32, 64), nn.MaxPool2d(2),
            make_block(64, 128), nn.MaxPool2d(2),
        )

        self.pool = nn.AdaptiveAvgPool2d(1) if pool_type == "GAP" else DeterministicGlobalMax()
        self.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x, mask=None):
        """按当前模型定义提取并汇总特征，输出未经sigmoid的logit。"""
        x = self.enc(x)
        x = self.pool(x).view(x.size(0), -1)
        return self.fc(x)

class ClasswiseFusion(nn.Module):
    """既有B分类头，保持Top16和18个独立融合参数，不新增方法。"""
    def __init__(self):
        """构造模型或数据对象，保持原参数与操作顺序。"""
        super().__init__()
        self.enc=PokemonModel(False,"GAP").enc
        self.channel_dropout=nn.Dropout2d(.5)
        self.classifier=nn.Conv2d(128,18,1,bias=True)
        self.fusion_logits=nn.Parameter(torch.zeros(18))
    def forward(self,x,mask=None):
        """每类在同一得分图上取全平均和Top16平均，再按sigmoid比例融合。"""
        maps=self.classifier(self.channel_dropout(self.enc(x))).flatten(2)
        global_scores=maps.mean(-1)
        local_scores=maps.topk(16,dim=-1,largest=True,sorted=True).values.mean(-1)
        alpha=torch.sigmoid(self.fusion_logits).reshape(1,-1)
        return (1-alpha)*global_scores+alpha*local_scores

class SharedFusion(ClasswiseFusion):
    """与已完成B仅融合参数粒度不同，18类共享一个可学习标量。"""
    def __init__(self):
        """建立原B结构，将18个初值0替换为一个初值0。"""
        super().__init__()
        self.fusion_logits = nn.Parameter(torch.zeros(1))
    def forward(self, x, mask=None):
        """沿用B的通道Dropout、类别得分图、全平均与Top16；alpha输入不用。"""
        maps = self.classifier(self.channel_dropout(self.enc(x))).flatten(2)
        global_scores = maps.mean(-1)
        local_scores = maps.topk(16, dim=-1, largest=True, sorted=True).values.mean(-1)
        alpha = torch.sigmoid(self.fusion_logits).reshape(1, 1)
        return (1 - alpha) * global_scores + alpha * local_scores

class ForegroundPool(nn.Module):
    """无参数前景加权特征平均；mask是输入先验，不参与梯度。"""
    def forward(self, features, mask):
        """把原128×128 alpha面积平均到特征网格，再按有效覆盖总量归一化。"""
        assert mask is not None and mask.ndim == 4 and mask.shape[1] == 1
        assert mask.shape[0] == features.shape[0]
        weights = F.adaptive_avg_pool2d(mask.detach().to(device=features.device, dtype=features.dtype),
                                       features.shape[-2:])
        total = weights.sum(dim=(2, 3), keepdim=True)
        assert bool(torch.isfinite(weights).all()) and bool((weights >= 0).all())
        assert bool((total > 0).all()), 'M必须大于0，不允许悄悄回退GAP'
        return (features * weights).sum(dim=(2, 3), keepdim=True) / total

class ForegroundModel(PokemonModel):
    """完整保留原Plain_GAP的enc和fc参数键，仅替换无参数pool。"""
    def __init__(self):
        """原分类器仍为Dropout(0.5)后接Linear，不新增模型参数。"""
        super().__init__(False, 'GAP')
        self.pool = ForegroundPool()
    def forward(self, x, mask=None):
        """前景平均后的128维特征进入原fc。"""
        features = self.enc(x)
        pooled = self.pool(features, mask).flatten(1)
        return self.fc(pooled)

def build_model(name, seed):
    """现场生成配对的未训练状态，不依赖任何旧checkpoint。"""
    assert name in TRAIN_MODEL_NAMES
    seed_everything(seed)
    reference = PokemonModel(False, "GAP")
    initial = cpu_state(reference)
    if name == "Residual_GAP":
        seed_everything(seed)  # 残差架构保持自己的原初始化规则。
        model = PokemonModel(True, "GAP")
    elif name in ("A_Shared", "B_Classwise"):
        model = SharedFusion() if name == "A_Shared" else ClasswiseFusion()
        model.enc.load_state_dict(reference.enc.state_dict())
        with torch.no_grad():
            model.classifier.weight.copy_(initial["fc.1.weight"][:, :, None, None])
            model.classifier.bias.copy_(initial["fc.1.bias"])
            model.fusion_logits.zero_()  # 初始全图/Top16各占0.5。
    else:
        model = ForegroundModel() if name == "G3_Foreground" else PokemonModel(False, "GAP" if name == "Plain_GAP" else "GMP")
        model.load_state_dict(initial, strict=True)
    return model

def stage_folder(kind, fold_id, name):
    """所有新增训练与预测均写入本次输出目录。"""
    return OUTPUT_DIR / "cv" / f"fold_{fold_id}" / name if kind == "cv" else OUTPUT_DIR / "full687" / name

def build_g2(kind, fold_id):
    """冻结本次GAP对应权重，只替换无参数pool；不更新BN。"""
    path = stage_folder(kind, fold_id, "Plain_GAP") / ("best.pt" if kind == "cv" else "final.pt")
    saved = torch.load(path, map_location="cpu", weights_only=False)
    assert saved["protocol_id"] == PROTOCOL_ID
    if kind == "cv":
        source_epoch = saved["best_record"]["epoch"]
        source_threshold = saved["best_record"]["threshold"]
    else:
        source_epoch, source_threshold = saved["epoch"], saved["threshold"]
    with torch.random.fork_rng(devices=[]):
        model = ForegroundModel()
        model.load_state_dict(saved["model_state"], strict=True)
    model.eval().requires_grad_(False)
    assert state_digest(model.state_dict()) == state_digest(saved["model_state"])
    source = {"path": relative_output(path), "sha256": hash_file(path),
              "source_epoch": int(source_epoch), "source_threshold": float(source_threshold),
              "state_sha256": state_digest(model.state_dict()), "protocol_id": PROTOCOL_ID}
    return model, source

def stage_indices(kind, fold_id):
    """沿用固定CV成员；full687不留验证集，也不参与终点选择。"""
    if kind == 'cv':
        fold = FOLDS[fold_id - 1]
        return fold['train'], fold['val'], fold['seed']
    assert kind == 'full687'
    return DEV_IDS, None, CONFIG['full_seed']


```markdown
## 训练损失 选择指标 阈值与最终指标

训练：当前train的正例权重为负例数/max(正例数,1)，使用BCEWithLogitsLoss反向更新网络；full687重新按687图计算。

选择：每轮验证集扫描九个全局阈值。Macro-F1优先，阈值并列比较Micro-F1，再并列取先遇到的较低阈值；epoch并列取更低验证BCE，再并列保持较早轮。五折最佳epoch和阈值分别取中位数。最终687图重训只用预定终点，不重新选“最佳”。

最终：模型与阈值冻结后报告历史留出集mAP、Macro/Micro-F1、逐类AP、TP/FP/FN。mAP是18类AP均值，不依赖单个分类阈值；AP比较同一类别下不同图片的正负排序。历史留出集已在探索中查看，不能称全新盲测。
```


In [ ]:
# =========================
# 5. 评价、选模与原训练步骤
# =========================
def safe_mean_average_precision(labels, probs):
    """逐类计算非插值AP后等权平均；正式划分18类均有正例。"""
    ap_scores = []
    for class_idx in range(labels.shape[1]):
        y_true = labels[:, class_idx]
        if y_true.sum() == 0:
            continue
        ap_scores.append(average_precision_score(y_true, probs[:, class_idx]))
    return float(np.mean(ap_scores)) if ap_scores else 0.0

def compute_f1_metrics(probs, labels, threshold):
    """用给定全局阈值计算Macro和Micro F1，不参与梯度。"""
    preds = (probs >= threshold).astype(int)
    return {
        "macro_f1": float(f1_score(labels, preds, average="macro", zero_division=0)),
        "micro_f1": float(f1_score(labels, preds, average="micro", zero_division=0)),
    }

def compute_test_detail_metrics(probs, labels, threshold):
    """在固定阈值下计算补充指标，不用于选择模型。"""
    preds = (probs >= threshold).astype(int)
    return {
        "macro_f1": float(f1_score(labels, preds, average="macro", zero_division=0)),
        "micro_f1": float(f1_score(labels, preds, average="micro", zero_division=0)),
        "precision": float(precision_score(labels, preds, average="macro", zero_division=0)),
        "recall": float(recall_score(labels, preds, average="macro", zero_division=0)),
        "class_f1": f1_score(labels, preds, average=None, zero_division=0).tolist(),
    }

def find_best_threshold(probs, labels, thresholds):
    """验证集Macro优先、Micro次优；完全并列保留较低阈值。"""
    best = {
        "threshold": float(thresholds[0]),
        "macro_f1": -1.0,
        "micro_f1": -1.0,
    }

    for threshold in thresholds:
        metrics = compute_f1_metrics(probs, labels, threshold)

        better_macro = metrics["macro_f1"] > best["macro_f1"] + 1e-12
        same_macro = abs(metrics["macro_f1"] - best["macro_f1"]) <= 1e-12
        better_micro = metrics["micro_f1"] > best["micro_f1"] + 1e-12

        if better_macro or (same_macro and better_micro):
            best = {
                "threshold": float(threshold),
                "macro_f1": metrics["macro_f1"],
                "micro_f1": metrics["micro_f1"],
            }

    return best

def choose_epoch(current,best):
    """与原代码相同：MacroF1优先，并列按更低val loss，不看MicroF1或mAP。"""
    return (best is None or current["macro_f1"]>best["macro_f1"]+1e-12 or
            (abs(current["macro_f1"]-best["macro_f1"])<=1e-12 and
             current["val_loss"]<best["val_loss"]-1e-12))

def threshold_grid(scores):
    """完整九阈值表；最终选择仍调用原find_best_threshold。"""
    selected=find_best_threshold(scores["probs"],scores["labels"],THRESHOLDS)
    table=pd.DataFrame([{"threshold":float(t),**compute_f1_metrics(scores["probs"],scores["labels"],t)} for t in THRESHOLDS])
    table["selected"]=table["threshold"].eq(selected["threshold"])
    return selected,table

def evaluate(model,indices,seed,criterion):
    """完整集合eval模式预测，不更新BN和参数。"""
    loader=make_loader(PairedDataset(indices),seed)
    model.eval();probs=[];targets=[];rows=[];loss_sum=0.;count=0
    with torch.inference_mode():
        for images,masks,y,ids in loader:
            images,masks,y=images.to(DEVICE),masks.to(DEVICE),y.to(DEVICE)
            logits=model(images,masks)
            loss_sum+=criterion(logits,y).item()*len(images);count+=len(images)
            probs.append(torch.sigmoid(logits).cpu().numpy());targets.append(y.cpu().numpy());rows.append(ids.numpy())
    probs,targets,rows=np.concatenate(probs),np.concatenate(targets),np.concatenate(rows)
    assert np.array_equal(targets,labels[rows])
    return {"probs":probs,"labels":targets,"row_ids":rows,"loss":loss_sum/count,
            "mAP":safe_mean_average_precision(targets,probs)}

def class_metrics(y,p,threshold):
    """保存所有类别，不只选提高的类别。"""
    truth=y.astype(bool);pred=p>=threshold
    tp=(truth&pred).sum(0);fp=(~truth&pred).sum(0);fn=(truth&~pred).sum(0)
    return pd.DataFrame({"class_name":CLASS_NAMES,"positive_count":truth.sum(0),
                         "AP":[average_precision_score(y[:,i],p[:,i]) for i in range(18)],
                         "TP":tp,"FP":fp,"FN":fn,
                         "precision":tp/np.maximum(tp+fp,1),"recall":tp/np.maximum(tp+fn,1),
                         "f1":2*tp/np.maximum(2*tp+fp+fn,1)})

def persist_predictions(scores,folder,prefix,threshold,identity):
    """按行号保存分数、标签及完整指标，支持独立复算。"""
    order=np.argsort(scores["row_ids"],kind="stable")
    rows=scores["row_ids"][order];y=scores["labels"][order];p=scores["probs"][order]
    np.savez_compressed(folder/f"{prefix}_predictions.npz",row_ids=rows,labels=y,probs=p,
                        class_names=np.asarray(CLASS_NAMES),threshold=np.asarray(threshold),
                        protocol_id=np.asarray(PROTOCOL_ID))
    table=metadata.set_index("row_id").loc[rows,["image_id"]].reset_index()
    for i,c in enumerate(CLASS_NAMES):
        table[f"label_{c}"]=y[:,i].astype(int);table[f"score_{c}"]=p[:,i]
        table[f"prediction_{c}"]=(p[:,i]>=threshold).astype(int)
    table.to_csv(folder/f"{prefix}_predictions.csv",index=False,encoding="utf-8-sig")
    metrics={"mAP":safe_mean_average_precision(y,p),**compute_f1_metrics(p,y,threshold),
             "weighted_BCE":scores["loss"],"sample_count":len(rows),"threshold":float(threshold)}
    save_json(folder/f"{prefix}_metrics.json",{"protocol_id":PROTOCOL_ID,**identity,**metrics})
    class_metrics(y,p,threshold).to_csv(folder/f"{prefix}_per_class.csv",index=False,encoding="utf-8-sig")
    return metrics

def train_one_epoch(model,loader,optimizer,criterion):
    """原train步骤，loss反传更新全部模型参数；阈值和mAP不参与梯度。"""
    model.train();total=0.;count=0
    for images,masks,y,_ in loader:
        images,masks,y=images.to(DEVICE),masks.to(DEVICE),y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss=criterion(model(images,masks),y)
        assert torch.isfinite(loss)
        loss.backward();optimizer.step()
        total+=loss.item()*len(images);count+=len(images)
    return total/count


```markdown
## 零更新小样本检查

先检查前景加权手算2.75、全1mask恢复GAP、mask无梯度、Top16融合手算19.5、参数量、共享初态、RGB及随机流配对、阈值/epoch并列规则、真实4张开发图的有限梯度及随机状态恢复。

本节没有optimizer.step。真实样本仅来自开发集，不先查看测试预测。check可以在CPU执行这些检查；正式训练仍要求CUDA及记录中的依赖版本。CPU小样本通过不等于在CPU上完整复现成绩。
```


In [ ]:
# =========================
# 6. 数据准备与零更新预检
# =========================
def prepare_data():
    """仅check/train显式调用，默认results不读取原始数据。"""
    global metadata, labels, INPUT_MANIFEST, CSV_SHA256, DEV_IDS, TEST_IDS, FOLDS, RGB_IMAGES, ALPHA_IMAGES
    metadata, labels, INPUT_MANIFEST, CSV_SHA256 = load_dataset()
    DEV_IDS, TEST_IDS, FOLDS = load_fixed_splits(labels)
    RGB_IMAGES, ALPHA_IMAGES = [], []
    for path in metadata["image_path"]:
        with Image.open(path) as image:
            RGB_IMAGES.append(image.convert("RGB").copy())
            ALPHA_IMAGES.append(image.getchannel("A").copy())
    print("数据核验通过：809图，687开发/122历史留出；五折成员固定。")

def run_preflight():
    """真实检查模型输入与梯度，但不执行优化更新，也不写训练产物。"""
    outer = {"python": random.getstate(), "numpy": np.random.get_state(),
             "cpu": torch.get_rng_state(), "cuda": torch.cuda.get_rng_state_all()}
    result = {"optimizer_steps": 0, "test_model_predictions": 0}
    try:
        features = torch.tensor([1., 2., 3., 4.]).reshape(1, 1, 2, 2).requires_grad_()
        mask = torch.tensor([0., 1., .5, .5]).reshape(1, 1, 2, 2).requires_grad_()
        pooled = ForegroundPool()(features, mask)
        assert pooled.item() == 2.75
        pooled.sum().backward()
        assert mask.grad is None
        assert torch.equal(features.grad.flatten(), torch.tensor([0., .5, .25, .25]))
        assert torch.equal(ForegroundPool()(features.detach(), torch.ones_like(mask)), features.detach().mean((2, 3), keepdim=True))
        result["foreground_manual"] = 2.75

        # 在真实A/B前向函数中用可手算得分图替换主干与分类器。
        for cls, expected_gradient in [(SharedFusion, 36.), (ClasswiseFusion, 2.)]:
            model = cls()
            model.enc = nn.Identity()
            model.channel_dropout = nn.Identity()
            model.classifier = nn.Identity()
            output = model(torch.arange(32.).reshape(1, 1, 1, 32).repeat(1, 18, 1, 1))
            assert torch.equal(output, torch.full((1, 18), 19.5))
            output.sum().backward()
            assert torch.equal(model.fusion_logits.grad, torch.full_like(model.fusion_logits, expected_gradient))
        result["fusion_manual"] = 19.5

        # 并列最大值的反传必须与原AdaptiveMaxPool一致。
        first = torch.tensor([2., 2., 1., 0.]).reshape(1, 1, 2, 2).requires_grad_()
        second = first.detach().clone().requires_grad_()
        a, b = DeterministicGlobalMax()(first), nn.AdaptiveMaxPool2d(1)(second)
        a.sum().backward(); b.sum().backward()
        assert torch.equal(a, b) and torch.equal(first.grad, second.grad)

        for training in (False, True):
            before = OriginalPairedDataset(DEV_IDS[:32], training, 42)
            after = PairedDataset(DEV_IDS[:32], training, 42)
            for index in range(32):
                rgb, label, row = before[index]
                paired_rgb, alpha, paired_label, paired_row = after[index]
                assert torch.equal(rgb, paired_rgb) and torch.equal(label, paired_label) and row == paired_row
                assert alpha.shape == (1, 128, 128) and alpha.sum() > 0
            assert torch.equal(before.augmentation_generator.get_state(), after.augmentation_generator.get_state())
        result["RGB_identical_examples_each_mode"] = 32

        # 两个batch检查样本顺序与增强结果，均使用第一折训练成员。
        old_data = OriginalPairedDataset(FOLDS[0]["train"], True, 42)
        new_data = PairedDataset(FOLDS[0]["train"], True, 42)
        old_loader = iter(make_loader(old_data, 42, True))
        new_loader = iter(make_loader(new_data, 42, True))
        for _ in range(2):
            x1, y1, ids1 = next(old_loader)
            x2, _, y2, ids2 = next(new_loader)
            assert torch.equal(x1, x2) and torch.equal(y1, y2) and torch.equal(ids1, ids2)
        result["paired_first_two_batches"] = True

        parameter_counts = {}
        x, alpha, y, _ = next(iter(make_loader(PairedDataset(FOLDS[0]["train"][:4]), 42)))
        criterion = nn.BCEWithLogitsLoss(pos_weight=compute_pos_weight(FOLDS[0]["train"]).to(DEVICE))
        reference = build_model("Plain_GAP", 42)
        for name in TRAIN_MODEL_NAMES:
            model = build_model(name, 42)
            parameter_counts[name] = sum(value.numel() for value in model.parameters())
            if name in ("Plain_GMP", "G3_Foreground"):
                assert state_digest(model.state_dict()) == state_digest(reference.state_dict())
            elif name in ("A_Shared", "B_Classwise"):
                assert state_digest(model.enc.state_dict()) == state_digest(reference.enc.state_dict())
                assert torch.equal(model.classifier.weight[:, :, 0, 0], reference.fc[1].weight)
                assert torch.equal(model.classifier.bias, reference.fc[1].bias)
            model = model.to(DEVICE).train()
            loss = criterion(model(x.to(DEVICE), alpha.to(DEVICE)), y.to(DEVICE))
            assert torch.isfinite(loss)
            loss.backward()
            assert all(value.grad is not None and bool(torch.isfinite(value.grad).all()) for value in model.parameters())
        assert parameter_counts == {"Plain_GAP": 290226, "Residual_GAP": 301234, "Plain_GMP": 290226,
                                    "A_Shared": 290227, "B_Classwise": 290244, "G3_Foreground": 290226}
        result["parameter_counts"] = parameter_counts

        # 验证前景推理冻结参数和BN；这里使用未训练GAP，仅检查数据链路。
        frozen = ForegroundModel().to(DEVICE)
        frozen.load_state_dict(reference.state_dict())
        frozen.eval().requires_grad_(False)
        digest = state_digest(frozen.state_dict())
        with torch.inference_mode():
            assert torch.isfinite(frozen(x.to(DEVICE), alpha.to(DEVICE))).all()
        assert state_digest(frozen.state_dict()) == digest
        assert not any(value.requires_grad for value in frozen.parameters())

        toy_y = np.asarray([[1, 0], [0, 1]])
        selected = find_best_threshold(np.zeros((2, 2)), toy_y, THRESHOLDS)
        assert selected["threshold"] == THRESHOLDS[0]
        best = {"macro_f1": .3, "val_loss": 1.0}
        assert not choose_epoch(best.copy(), best)
        assert choose_epoch({"macro_f1": .3, "val_loss": .9}, best)
        assert not choose_epoch({"macro_f1": .29, "val_loss": .1}, best)
        result["selection_ties"] = True

        data = PairedDataset(FOLDS[0]["train"][:4], True, 42)
        generator = torch.Generator().manual_seed(1042)
        saved = capture_random(generator, data)
        expected_rgb = data[0][0]
        expected_random = (random.random(), np.random.rand(), torch.rand(3), torch.rand(3, generator=generator))
        restore_random(saved, generator, data)
        actual_rgb = data[0][0]
        actual_random = (random.random(), np.random.rand(), torch.rand(3), torch.rand(3, generator=generator))
        assert torch.equal(expected_rgb, actual_rgb)
        assert expected_random[:2] == actual_random[:2]
        assert all(torch.equal(a, b) for a, b in zip(expected_random[2:], actual_random[2:]))
        result["RNG_roundtrip"] = True
        result["passed"] = True
        return result
    finally:
        random.setstate(outer["python"]); np.random.set_state(outer["numpy"])
        torch.set_rng_state(outer["cpu"]); torch.cuda.set_rng_state_all(outer["cuda"])


```markdown
## 分组训练与断点恢复

只有MODE=train进入本节运行函数。每折六个训练模型固定100轮，先保存best再评价G2；G2不重新挑epoch。全部五折结果形成后冻结E/τ中位数，六个模型分别以全部687图从零训练到自身E，随后统一冻结七个评价条件。

初始化、模型、优化器、BN和随机状态均保存。已完成轨迹直接读取；中断读取last继续，不能用初始种子覆盖已运行状态。修改代码或配置后使用新的输出目录，不把新旧运行混合。
```


In [ ]:
# =========================
# 7. 实验身份、训练与选择
# =========================
def freeze_json(path, payload, key):
    """已有冻结文件必须与当前请求一致，不覆盖另一实验的选择。"""
    digest = hashlib.sha256(json.dumps(payload, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()
    result = {**payload, key: digest}
    if path.exists():
        assert json.loads(path.read_text(encoding="utf-8")) == result, "已有结果与当前协议不同，请使用新的输出目录。"
    else:
        save_json(path, result)
    return result

def initialize_run(preflight):
    """预检通过后才建立训练目录，生成公开版本自己的协议身份。"""
    global PROTOCOL_ID
    assert MODE == "train" and preflight["passed"]
    assert DEVICE.type == "cuda", "正式36条训练使用CUDA；CPU请先使用check模式。"
    payload = {"schema": "pokemon-cnn-public-v1", "config": CONFIG, "class_names": CLASS_NAMES,
               "model_names": ALL_MODEL_NAMES, "trained_model_names": TRAIN_MODEL_NAMES,
               "environment": ENVIRONMENT, "csv_sha256": CSV_SHA256,
               "semantic_sha256": DATA_MANIFEST["semantic_sha256"],
               "image_manifest_sha256": hash_file(ROOT / "config/data_manifest.json"),
               "splits": FIXED_SPLITS, "implementation_sha256": implementation_hash(),
               "selection_uses_test": False, "test_is_historical_holdout": True}
    digest = hashlib.sha256(json.dumps(payload, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()
    if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()) and not (OUTPUT_DIR / "protocol.json").exists():
        raise ValueError("输出目录已有未知文件，请设置新的POKEMON_OUTPUT_DIR。")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    frozen = freeze_json(OUTPUT_DIR / "protocol.json", payload, "protocol_id")
    assert frozen["protocol_id"] == digest
    PROTOCOL_ID = digest
    save_json(OUTPUT_DIR / "preflight_checks.json", preflight)

def run_stage(kind,fold_id,name,max_epochs,final_threshold=None,selection_plan_id=None):
    """CV按原规则选择；full687仅训练预先固定轮数，不验证或测试。"""
    assert MODE == "train", "只有train模式允许更新模型。"
    train_ids,val_ids,seed=stage_indices(kind,fold_id)
    run_dir=stage_folder(kind,fold_id,name);last_path=run_dir/"last.pt"
    identity={"protocol_id":PROTOCOL_ID,"kind":kind,"fold":fold_id,"model":name,"seed":seed,
              "max_epochs":int(max_epochs),"threshold":final_threshold,"selection_plan_id":selection_plan_id,
              "train_ids":train_ids.tolist(),"val_ids":None if val_ids is None else val_ids.tolist()}
    signature=hashlib.sha256(json.dumps(identity,sort_keys=True).encode("utf-8")).hexdigest()
    snapshot=None
    if last_path.exists():
        snapshot=torch.load(last_path,map_location="cpu",weights_only=False)
        assert snapshot["signature"]==signature and snapshot["device"]==str(DEVICE)
        if snapshot["completed"]:
            return json.loads((run_dir/"result.json").read_text(encoding="utf-8"))
    else:
        assert not run_dir.exists() or not any(run_dir.iterdir()),"未知产物不能覆盖"
    run_dir.mkdir(parents=True,exist_ok=True)
    model=build_model(name,seed).to(DEVICE)
    initial_digest=state_digest(model.state_dict())
    optimizer=torch.optim.Adam(model.parameters(), lr=CONFIG["learning_rate"],
                               betas=tuple(CONFIG["adam_betas"]), eps=CONFIG["adam_eps"],
                               weight_decay=CONFIG["weight_decay"], amsgrad=CONFIG["amsgrad"])
    criterion=nn.BCEWithLogitsLoss(pos_weight=compute_pos_weight(train_ids).to(DEVICE),reduction="mean")
    data=PairedDataset(train_ids,True,seed)
    generator=torch.Generator().manual_seed(seed+CONFIG["loader_rng_offset"])
    loader=make_loader(data,seed,True,generator)
    if snapshot is None:
        seed_everything(seed+CONFIG["train_rng_offset"])
        snapshot={"protocol_id":PROTOCOL_ID,"signature":signature,"identity":identity,"device":str(DEVICE),
                  "epoch":0,"completed":False,"model_state":cpu_state(model),"optimizer_state":optimizer.state_dict(),
                  "random_state":capture_random(generator,data),"history":[],"best_record":None,"best_state":None,
                  "initial_state_sha256":initial_digest,"started_at":now_local()}
        save_checkpoint(run_dir/"initial.pt",snapshot)
        save_checkpoint(last_path,snapshot)
    else:
        assert snapshot["initial_state_sha256"]==initial_digest
        model.load_state_dict(snapshot["model_state"])
        optimizer.load_state_dict(snapshot["optimizer_state"])
        restore_random(snapshot["random_state"],generator,data)
    save_json(run_dir/"config.json",{**identity,"signature":signature,
              "pos_weight":compute_pos_weight(train_ids).tolist(),"initial_state_sha256":initial_digest})
    history=snapshot["history"];best_record=snapshot["best_record"];best_state=snapshot["best_state"]
    for epoch in range(snapshot["epoch"]+1,max_epochs+1):
        started=time.perf_counter()
        train_loss=train_one_epoch(model,loader,optimizer,criterion)
        row={"epoch":epoch,"train_loss":train_loss,"learning_rate":optimizer.param_groups[0]["lr"]}
        if kind=="cv":
            scores=evaluate(model,val_ids,seed,criterion)
            selected=find_best_threshold(scores["probs"],scores["labels"],THRESHOLDS)
            current={"epoch":epoch,"threshold":selected["threshold"],"macro_f1":selected["macro_f1"],
                     "micro_f1":selected["micro_f1"],"val_loss":scores["loss"],"mAP":scores["mAP"]}
            if choose_epoch(current,best_record):
                best_record=current.copy();best_state=cpu_state(model)
            row.update({"val_loss":scores["loss"],"val_mAP":scores["mAP"],
                        "val_macro_f1":selected["macro_f1"],"val_micro_f1":selected["micro_f1"],
                        "val_threshold":selected["threshold"],"best_epoch":best_record["epoch"]})
        row["epoch_seconds"]=time.perf_counter()-started;history.append(row)
        snapshot={"protocol_id":PROTOCOL_ID,"signature":signature,"identity":identity,"device":str(DEVICE),
                  "epoch":epoch,"completed":False,"model_state":cpu_state(model),"optimizer_state":optimizer.state_dict(),
                  "random_state":capture_random(generator,data),"history":history,"best_record":best_record,
                  "best_state":best_state,"initial_state_sha256":initial_digest,"started_at":snapshot["started_at"]}
        save_checkpoint(last_path,snapshot)
        pd.DataFrame(history).to_csv(run_dir/"epoch_log.csv",index=False,encoding="utf-8-sig")
        save_json(OUTPUT_DIR/"progress.json",{"kind":kind,"fold":fold_id,"model":name,"epoch":epoch,
                  "target_epochs":max_epochs,"last_train_loss":train_loss,"best_record":best_record,"time":now_local()})
        if epoch==1 or epoch%10==0 or epoch==max_epochs:
            print(kind,fold_id,name,f"{epoch}/{max_epochs}","train_loss",round(train_loss,6),
                  "best",None if best_record is None else best_record["epoch"],flush=True)
    steps=int(np.ceil(len(train_ids)/CONFIG["batch_size"]))*max_epochs
    assert len(optimizer.state)==len(list(model.parameters()))
    assert {float(v["step"]) for v in optimizer.state.values()}=={float(steps)}
    if kind=="cv":
        model.load_state_dict(best_state)
        scores=evaluate(model,val_ids,seed,criterion)
        selected,grid=threshold_grid(scores)
        assert selected["threshold"]==best_record["threshold"]
        assert abs(selected["macro_f1"]-best_record["macro_f1"])<1e-12
        assert abs(scores["mAP"]-best_record["mAP"])<1e-12
        grid.to_csv(run_dir/"threshold_grid.csv",index=False,encoding="utf-8-sig")
        save_checkpoint(run_dir/"best.pt",{"protocol_id":PROTOCOL_ID,"identity":identity,
                        "model_state":best_state,"best_record":best_record})
        persist_predictions(scores,run_dir,"val",best_record["threshold"],
                            {"model":name,"fold":fold_id,"best_epoch":best_record["epoch"]})
    else:
        save_checkpoint(run_dir/"final.pt",{"protocol_id":PROTOCOL_ID,"identity":identity,
                        "model_state":snapshot["model_state"],"epoch":max_epochs,"threshold":final_threshold,
                        "selection_plan_id":selection_plan_id})
    end_rng=snapshot["random_state"]
    result={"protocol_id":PROTOCOL_ID,"kind":kind,"fold":fold_id,"model":name,"seed":seed,
            "epochs_completed":max_epochs,"optimizer_updates":steps,"best_record":best_record,
            "parameter_count":sum(v.numel() for v in model.parameters()),
            "initial_state_sha256":initial_digest,
            "loader_end_sha256":hashlib.sha256(end_rng["loader"].numpy().tobytes()).hexdigest(),
            "augmentation_end_sha256":hashlib.sha256(end_rng["augmentation"].numpy().tobytes()).hexdigest(),
            "epoch_seconds":float(sum(r["epoch_seconds"] for r in history)),
            "started_at":snapshot["started_at"],"finished_at":now_local(),"completed":True}
    save_json(run_dir/"result.json",result)
    snapshot["completed"]=True;save_checkpoint(last_path,snapshot)
    return result

def run_g2_validation(fold):
    """只对冻结GAP折内权重做前景推理，不训练、不重新选择epoch。"""
    folder=stage_folder("cv",fold["fold"],"G2_Foreground")
    folder.mkdir(parents=True,exist_ok=True)
    result_path=folder/"result.json"
    if result_path.exists():
        saved=json.loads(result_path.read_text(encoding="utf-8"))
        assert saved["protocol_id"]==PROTOCOL_ID and saved["completed"]
        assert hash_file(OUTPUT_DIR / saved["source"]["path"])==saved["source"]["sha256"]
        return saved
    model,source=build_g2("cv",fold["fold"])
    model=model.to(DEVICE)
    before=state_digest(model.state_dict())
    criterion=nn.BCEWithLogitsLoss(pos_weight=compute_pos_weight(fold["train"]).to(DEVICE))
    scores=evaluate(model,fold["val"],fold["seed"],criterion)
    assert state_digest(model.state_dict())==before
    selected,grid=threshold_grid(scores)
    grid.to_csv(folder/"threshold_grid.csv",index=False,encoding="utf-8-sig")
    identity={"model":"G2_Foreground","fold":fold["fold"],"source_epoch":source["source_epoch"],"training_updates":0}
    persist_predictions(scores,folder,"val",selected["threshold"],identity)
    persist_predictions(scores,folder,"val_fixed_source_threshold",source["source_threshold"],identity)
    result={"protocol_id":PROTOCOL_ID,"model":"G2_Foreground","fold":fold["fold"],
            "seed":fold["seed"],"source":source,"optimizer_updates":0,"epochs_completed":0,
            "best_record":{"epoch":source["source_epoch"],"threshold":selected["threshold"],
                           "macro_f1":selected["macro_f1"],"micro_f1":selected["micro_f1"],
                           "val_loss":scores["loss"],"mAP":scores["mAP"]},
            "epoch_is_inherited_from_GAP":True,"BN_unchanged":True,"completed":True}
    save_json(result_path,result)
    return result

def run_cv_and_g2_validation():
    """本次自行完成30条折内训练；G2继承该折GAP已选权重。"""
    records, pairing = [], []
    for fold in FOLDS:
        reference_result = None
        for name in TRAIN_MODEL_NAMES:
            result = run_stage("cv", fold["fold"], name, CONFIG["epochs"])
            if name == "Plain_GAP":
                reference_result = result
            for key in ("loader_end_sha256", "augmentation_end_sha256"):
                assert result[key] == reference_result[key], f"{name}的数据随机流未配对：{key}"
            best = result["best_record"]
            records.append({"model": name, "fold": fold["fold"], "seed": fold["seed"],
                            "best_epoch": best["epoch"], "threshold": best["threshold"],
                            "macro_f1": best["macro_f1"], "micro_f1": best["micro_f1"], "mAP": best["mAP"],
                            "val_loss": best["val_loss"], "epoch_seconds": result["epoch_seconds"]})
            pairing.append({"model": name, "fold": fold["fold"], "matches_Plain_GAP": True})
        result = run_g2_validation(fold)
        best = result["best_record"]
        records.append({"model": "G2_Foreground", "fold": fold["fold"], "seed": fold["seed"],
                        "best_epoch": best["epoch"], "threshold": best["threshold"],
                        "macro_f1": best["macro_f1"], "micro_f1": best["micro_f1"], "mAP": best["mAP"],
                        "val_loss": best["val_loss"], "epoch_seconds": 0.0})
        save_json(OUTPUT_DIR / "paired_input_checks.json", pairing)
    table = pd.DataFrame(records)
    assert len(table) == 35 and table.groupby("model").size().eq(5).all()
    table.to_csv(OUTPUT_DIR / "cv_fold_results.csv", index=False, encoding="utf-8-sig")
    return table

def freeze_selections(table):
    """所有选择仅来自五折；G2继承本次GAP终点并自行选验证阈值。"""
    plans, aggregate = {}, []
    for name in ALL_MODEL_NAMES:
        rows = table.loc[table["model"].eq(name)]
        assert len(rows) == 5
        epoch = int(np.median(rows["best_epoch"].to_numpy()))
        threshold = float(np.median(rows["threshold"].to_numpy()))
        if name == "G2_Foreground":
            epoch = plans["Plain_GAP"]["epochs"]
            source_paths = [stage_folder("cv", fold["fold"], "Plain_GAP") / "best.pt" for fold in FOLDS]
        else:
            source_paths = [stage_folder("cv", fold["fold"], name) / "best.pt" for fold in FOLDS]
        plans[name] = {"epochs": epoch, "threshold": threshold,
                       "cv_sources": [{"path": relative_output(path), "sha256": hash_file(path)} for path in source_paths]}
        if name == "G2_Foreground":
            plans[name].update({"training_updates": 0, "source_model": "Plain_GAP",
                                "source_threshold": plans["Plain_GAP"]["threshold"]})
        row = {"model": name, "final_epochs": epoch, "final_threshold": threshold}
        for metric in ("macro_f1", "micro_f1", "mAP"):
            row["cv_" + metric + "_mean"] = float(rows[metric].mean())
            row["cv_" + metric + "_std"] = float(rows[metric].std(ddof=1))
        aggregate.append(row)
    pd.DataFrame(aggregate).to_csv(OUTPUT_DIR / "cv_aggregate.csv", index=False, encoding="utf-8-sig")
    return freeze_json(OUTPUT_DIR / "retrain_selection.json",
                       {"protocol_id": PROTOCOL_ID, "models": plans,
                        "dev_ids": DEV_IDS.tolist(), "test_ids": TEST_IDS.tolist(), "selection_uses_test": False},
                       "selection_plan_id")

def run_full687(plan):
    """六种方案从零训练全部687图，只使用预定终点，不留验证集。"""
    results = []
    for name in TRAIN_MODEL_NAMES:
        choice = plan["models"][name]
        results.append(run_stage("full687", 0, name, choice["epochs"], choice["threshold"], plan["selection_plan_id"]))
    return results

def run_all():
    """完整30CV加6full；完成记录可复用，中断按last恢复。"""
    assert MODE == "train"
    started = now_local()
    table = run_cv_and_g2_validation()
    plan = freeze_selections(table)
    full_results = run_full687(plan)
    result = {"protocol_id": PROTOCOL_ID, "selection_plan_id": plan["selection_plan_id"],
              "CV_trajectories": 30, "full_trajectories": 6, "training_trajectories": 36,
              "full_results": full_results, "started_at": started, "finished_at": now_local(), "completed": True}
    save_json(OUTPUT_DIR / "training_completion.json", result)
    return result


```markdown
## 最终冻结和统一评价

七个评价条件的权重及阈值先共同冻结，再计算train与历史留出集预测。G2主阈值来自自身验证结果，另保存沿用GAP阈值的诊断；诊断即使更高也不替换主结果。

结果包含7组主表、35条折内记录、18类AP、标签级误报漏报、成对差值与融合参数。TP/FP/FN按图片—类别对计算，不是错误图片数。新结果保存在outputs，附带历史结果始终在results/reference_20261027。
```


In [ ]:
# =========================
# 8. 冻结模型、最终评价与结果展示
# =========================
def confusion_totals(scores, threshold):
    """统计全部图片—类别判断的TP/FP/FN。"""
    truth = scores["labels"].astype(bool)
    pred = scores["probs"] >= threshold
    return {"TP": int((truth & pred).sum()), "FP": int((~truth & pred).sum()), "FN": int((truth & ~pred).sum())}

def finalize_results(run_result):
    """统一冻结七个条件后再预测，不根据测试调整参数或阈值。"""
    assert MODE == "train" and run_result["completed"]
    plan = json.loads((OUTPUT_DIR / "retrain_selection.json").read_text(encoding="utf-8"))
    assert plan["protocol_id"] == PROTOCOL_ID and not plan["selection_uses_test"]
    entries = {}
    for name in ALL_MODEL_NAMES:
        source_name = "Plain_GAP" if name == "G2_Foreground" else name
        path = stage_folder("full687", 0, source_name) / "final.pt"
        saved = torch.load(path, map_location="cpu", weights_only=False)
        assert saved["protocol_id"] == PROTOCOL_ID
        assert saved["selection_plan_id"] == plan["selection_plan_id"]
        assert saved["epoch"] == plan["models"][name]["epochs"]
        assert saved["identity"]["train_ids"] == DEV_IDS.tolist()
        if name != "G2_Foreground":
            assert saved["threshold"] == plan["models"][name]["threshold"]
        entries[name] = {"path": relative_output(path), "sha256": hash_file(path),
                         "epochs": plan["models"][name]["epochs"], "threshold": plan["models"][name]["threshold"]}
    frozen = freeze_json(OUTPUT_DIR / "final_models_frozen.json",
                         {"protocol_id": PROTOCOL_ID, "selection_plan_id": plan["selection_plan_id"],
                          "models": entries, "test_ids": TEST_IDS.tolist()}, "evaluation_id")
    if (OUTPUT_DIR / "completion.json").exists():
        complete = json.loads((OUTPUT_DIR / "completion.json").read_text(encoding="utf-8"))
        assert complete["evaluation_id"] == frozen["evaluation_id"]
        for relative_path, digest in complete["summary_files_sha256"].items():
            assert hash_file(OUTPUT_DIR / relative_path) == digest
        return complete
    save_json(OUTPUT_DIR / "test_started.json", {"evaluation_id": frozen["evaluation_id"], "time": now_local()})
    aggregate = pd.read_csv(OUTPUT_DIR / "cv_aggregate.csv", float_precision="round_trip").set_index("model")
    summary, counts, fusion, g2_diagnostics = [], [], [], []
    class_ap = pd.DataFrame({"class_name": CLASS_NAMES, "test_positive_count": labels[TEST_IDS].sum(0).astype(int)})
    criterion = nn.BCEWithLogitsLoss(pos_weight=compute_pos_weight(DEV_IDS).to(DEVICE))
    for name in ALL_MODEL_NAMES:
        entry = entries[name]
        path = OUTPUT_DIR / entry["path"]
        assert hash_file(path) == entry["sha256"]
        if name == "G2_Foreground":
            model, source = build_g2("full687", 0)
        else:
            model = build_model(name, CONFIG["full_seed"])
            model.load_state_dict(torch.load(path, map_location="cpu", weights_only=False)["model_state"])
        model = model.to(DEVICE).eval()
        before = state_digest(model.state_dict())
        folder = stage_folder("full687", 0, name)
        folder.mkdir(parents=True, exist_ok=True)
        if name == "G2_Foreground":
            save_json(folder / "source_binding.json", source)
        row = {"model": name, **aggregate.loc[name].to_dict(),
               "parameter_count": sum(value.numel() for value in model.parameters()),
               "uses_alpha": name in ("G2_Foreground", "G3_Foreground")}
        for split, ids in [("train", DEV_IDS), ("test", TEST_IDS)]:
            scores = evaluate(model, ids, CONFIG["full_seed"], criterion)
            metrics = persist_predictions(scores, folder, split, entry["threshold"],
                                          {"model": name, "evaluation_id": frozen["evaluation_id"]})
            totals = confusion_totals(scores, entry["threshold"])
            row.update({split + "_" + key: value for key, value in metrics.items() if key not in ("threshold", "sample_count")})
            row.update({split + "_" + key: value for key, value in totals.items()})
            counts.append({"model": name, "split": split, **totals, "threshold": entry["threshold"],
                           "true_labels_per_image": float(scores["labels"].sum(1).mean()),
                           "predicted_labels_per_image": float((scores["probs"] >= entry["threshold"]).sum(1).mean())})
            if split == "test":
                class_ap[name] = class_metrics(scores["labels"], scores["probs"], entry["threshold"])["AP"]
            if name == "G2_Foreground":
                fixed_threshold = plan["models"]["Plain_GAP"]["threshold"]
                diagnostic = persist_predictions(scores, folder, split + "_fixed_source_threshold", fixed_threshold,
                                                  {"model": name, "diagnostic_only": True})
                g2_diagnostics.append({"split": split, **diagnostic, **confusion_totals(scores, fixed_threshold)})
        assert state_digest(model.state_dict()) == before, "评价不得更新权重或BN。"
        if name in ("A_Shared", "B_Classwise"):
            theta = model.fusion_logits.detach().cpu().numpy().reshape(-1)
            beta = torch.sigmoid(model.fusion_logits.detach()).cpu().numpy().reshape(-1)
            for index, class_name in enumerate(CLASS_NAMES):
                j = 0 if name == "A_Shared" else index
                fusion.append({"model": name, "class_name": class_name, "learned_parameter_count": len(theta),
                               "parameter_scope": "shared_all_classes" if name == "A_Shared" else "per_class",
                               "theta": float(theta[j]), "global_weight": float(1 - beta[j]), "top16_weight": float(beta[j])})
        summary.append(row)
    summary = pd.DataFrame(summary)
    summary.to_csv(OUTPUT_DIR / "all_schemes_summary.csv", index=False, encoding="utf-8-sig")
    class_ap.to_csv(OUTPUT_DIR / "all_class_test_AP.csv", index=False, encoding="utf-8-sig")
    count_table = pd.DataFrame(counts)
    count_table.to_csv(OUTPUT_DIR / "confusion_counts.csv", index=False, encoding="utf-8-sig")
    count_table.loc[count_table["split"].eq("test")].to_csv(OUTPUT_DIR / "test_confusion_counts.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(fusion).to_csv(OUTPUT_DIR / "fusion_weights.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(g2_diagnostics).to_csv(OUTPUT_DIR / "G2_fixed_source_threshold_diagnostics.csv", index=False, encoding="utf-8-sig")
    lookup = summary.set_index("model")
    planned_pairs = [("A_Shared", "B_Classwise"), ("A_Shared", "Plain_GAP"),
                     ("B_Classwise", "Plain_GAP"), ("G2_Foreground", "Plain_GAP"),
                     ("G3_Foreground", "Plain_GAP"), ("G3_Foreground", "G2_Foreground")]
    all_pairs = [(candidate, baseline) for index, candidate in enumerate(ALL_MODEL_NAMES)
                 for baseline in ALL_MODEL_NAMES[:index]]
    for filename, comparisons in [("planned_contrasts.csv", planned_pairs), ("all_pairwise_deltas.csv", all_pairs)]:
        pairs = []
        for candidate, baseline in comparisons:
            record = {"candidate": candidate, "baseline": baseline}
            for key in ["test_mAP", "test_macro_f1", "test_micro_f1", "test_TP", "test_FP", "test_FN"]:
                record[key + "_delta"] = float(lookup.loc[candidate, key] - lookup.loc[baseline, key])
            pairs.append(record)
        pd.DataFrame(pairs).to_csv(OUTPUT_DIR / filename, index=False, encoding="utf-8-sig")
    expected_files = ["all_schemes_summary.csv", "cv_fold_results.csv", "cv_aggregate.csv", "all_class_test_AP.csv",
                      "test_confusion_counts.csv", "fusion_weights.csv", "G2_fixed_source_threshold_diagnostics.csv",
                      "planned_contrasts.csv", "all_pairwise_deltas.csv"]
    completion = {"protocol_id": PROTOCOL_ID, "evaluation_id": frozen["evaluation_id"],
                  "training_trajectories": 36, "CV_trajectories": 30, "full_trajectories": 6,
                  "conditions": 7, "completed": True, "finished_at": now_local(),
                  "summary_files_sha256": {name: hash_file(OUTPUT_DIR / name) for name in expected_files}}
    save_json(OUTPUT_DIR / "completion.json", completion)
    return completion

def display_summary(folder, reference=False):
    """默认显示历史参考；新结果必须显式从本次输出读取，不混淆来源。"""
    if reference:
        for filename, expected in REFERENCE["reference_tables"].items():
            assert hash_file(folder / filename) == expected, "参考结果表被修改。"
        print("2026-10-27已完成实验的参考结果；公开整理版本未重跑完整训练。")
    else:
        print("本次公开版本运行结果：", PROTOCOL_ID)
    table = pd.read_csv(folder / "all_schemes_summary.csv", float_precision="round_trip")
    display(table[["model", "final_epochs", "final_threshold", "parameter_count", "uses_alpha",
                   "test_mAP", "test_macro_f1", "test_micro_f1"]])
    display(table[["model", "cv_mAP_mean", "cv_mAP_std", "cv_macro_f1_mean", "cv_macro_f1_std",
                   "cv_micro_f1_mean", "cv_micro_f1_std"]])
    positions = np.arange(len(table))
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(positions - .18, table["cv_mAP_mean"], .36, yerr=table["cv_mAP_std"], label="CV mean +/- SD")
    ax.bar(positions + .18, table["test_mAP"], .36, label="Historical holdout")
    ax.set_xticks(positions, table["model"], rotation=25, ha="right")
    ax.set_ylabel("mAP"); ax.legend(); fig.tight_layout(); plt.show()
    print("7种评价条件 = 6种训练方案 + 1种G2同权重推理干预。")
    if reference:
        print("G2参考留出集最高，但五折下降；G3相对GMP小幅改善，伴随误报与漏报取舍。")
    return table


```markdown
## 执行入口

保持MODE=results可直接查看参考结果。准备数据后改为check，检查成功再决定是否改为train。完整训练会运行六种方案的五折与全开发集流程，G2只做推理。

代码不会上传GitHub、下载数据或自动安装依赖。结果目录不可覆盖results/reference_20261027。Notebook发布时清空执行输出，避免机器路径和训练日志混入代码。
```


In [ ]:
# =========================
# 9. 按明确模式执行
# =========================
if MODE == "results":
    SUMMARY = display_summary(REFERENCE_DIR, reference=True)
else:
    prepare_data()
    PREFLIGHT_RESULT = run_preflight()
    print("零更新检查：", PREFLIGHT_RESULT)
    if MODE == "train":
        initialize_run(PREFLIGHT_RESULT)
        RUN_RESULT = run_all()
        COMPLETION = finalize_results(RUN_RESULT)
        SUMMARY = display_summary(OUTPUT_DIR, reference=False)
    else:
        print("check完成：未进行optimizer.step，未进行历史测试集模型预测，未创建训练输出。")
